# C02w2 — Five-case stronger uniqueness-control pilot
Run in a fresh kernel. Regenerates from the original passages using the frozen model, prompt and human prefix curves. Selects the five new cases with the largest **absolute final excess over their supported upper target** (ties: case ID). Selection uses no detector scores.

Pilot changes: begin downward control above the human median; multiply downward token thresholds by 0.25; double the existing expansion/frontier budgets. Retain the 0.05 word-path likelihood floor, upward controller, warmup, candidate cap, EOS behavior and runaway checks. This is a bundled feasibility pilot, not an ablation. Unsupported prefixes remain unsupported. No final 54-text files are modified and no downstream stages are run here. Review fidelity and repetition before adopting outputs.


In [ ]:
# Export every figure to reproduction/figures as well as the run directory.
from pathlib import Path as _FigurePath
def _export_figure(figure, filename, *args, **kwargs):
    original = _FigurePath(filename)
    original.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(original, *args, **kwargs)
    figure_dir = ROOT / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    try:
        suffix = original.relative_to(OUT).with_suffix("").as_posix().replace("/", "_")
    except ValueError:
        suffix = original.stem
    target = figure_dir / ("C02w2_" + suffix + original.suffix)
    figure.savefig(target, *args, **kwargs)

from pathlib import Path
import os, json, hashlib, math, time, uuid, inspect, copy, gc, re
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd()/'results').exists() else Path.cwd().parent
PARENT = ROOT / 'results/causal/C02r_discovery_gated_punctuation'
BASE = ROOT / 'results/causal/C02d2_fz118_ghostbuster_raid'
PRIOR = ROOT / 'results/causal/C02e2_raid_feature_guided'
PREV = ROOT/'results/causal/C02s_relative_likelihood_sensitivity'
ARMS = {'robust_005': .05}
BOOTSTRAPS = 3000
SELECTION_SEED = 20260906
LENGTH_RATIO = 1.25
MEAN_DISTANCE_TOLERANCE = {'wps_mean': .5, 'cps_mean': 3.}
ARTIFICIAL_SD_SCALE = .5
BUMP_P_MIN = .10
normalize = lambda text: str(text).translate(str.maketrans({'\n':' ', '\r':' '}))
def digest(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()
def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False))
    os.replace(tmp, path)
def freeze(path, obj):
    if path.exists():
        assert json.loads(path.read_text()) == obj, f'Contract changed: use a new OUT ({path}).'
    else:
        write_json(path, obj)
def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def savefig(fig, stem):
    for ext in ('png', 'pdf'):
        _export_figure(fig,OUT / f'{stem}.{ext}', dpi=300, bbox_inches='tight')

parent_contract = json.loads((PARENT/'frozen_contract.json').read_text())
selection = pd.read_csv(PARENT/'selection.csv')
selection = selection.loc[selection.partition.eq('discovery')].copy()
assert len(selection) == 54 and selection.case_id.is_unique
selected_ids = set(selection.case_id)
cases = [copy.deepcopy(c) for c in parent_contract['cases'] if c['case_id'] in selected_ids]
assert {c['case_id'] for c in cases} == selected_ids
assert all(c['partition'] == 'discovery' for c in cases)
MODEL_ID = parent_contract['model']
MODEL_REVISION = parent_contract['resolved_revision']
assert MODEL_REVISION and MODEL_REVISION != 'main'
WARMUP_FRACTION = parent_contract['warmup_fraction']
MIN_PREFIX_SUPPORT = parent_contract['minimum_prefix_support']
THETA_SCHEDULES = parent_contract['theta_schedules']
MAX_COMPLETED_CANDIDATES = parent_contract['max_completed_candidates']
display(selection.groupby('domain').size().rename('discovery_cases'))
print('Only discovery IDs loaded:', len(cases))


LATEST=PREV/'wps_repair_v1/downstream_v1/hyphen_repair_v1/texts.csv'
known=set(pd.read_csv(LATEST).case_id)
ranking=[]
for c in cases:
    if c['case_id'] in known:continue
    p=PREV/c['case_id']/'ratio_005/state.json'
    state=json.loads(p.read_text())['state']
    assert state['status']=='completed_eos', c['case_id']
    t=state['trajectory'][-1];r=t['reference_after']
    if r['supported'] and t['k_after']>r['H_a10']:
        ranking.append(dict(case_id=c['case_id'],domain=c['domain'],old_unique=t['k_after'],old_upper=r['H_a10'],excess=t['k_after']-r['H_a10']))
ranking=pd.DataFrame(ranking).sort_values(['excess','case_id'],ascending=[False,True]).head(5)
assert len(ranking)==5,'Need five supported above-band new cases'
lookup={c['case_id']:c for c in cases}
cases=[lookup[cid] for cid in ranking.case_id]
selected_ids=set(ranking.case_id)
notebook_path=ROOT/'experiments/C02w2_regenerate_top5.ipynb'
NOTEBOOK_CODE_HASH=digest([c['source'] for c in json.loads(notebook_path.read_text())['cells'] if c['cell_type']=='code'])
input_hashes={str(p.relative_to(ROOT)):file_hash(p) for p in [PARENT/'frozen_contract.json',LATEST,*[PREV/c['case_id']/'ratio_005/state.json' for c in cases]]}
PILOT=dict(version='stronger_downward_v1',downward_theta_multiplier=.25,trigger='human_median',ratio_floor=.05,selection='top5_absolute_supported_excess',inputs=input_hashes,code=NOTEBOOK_CODE_HASH)
OUT=ROOT/'results/causal/C02w2_regenerate_top5'/digest(PILOT)[:12]
OUT.mkdir(parents=True,exist_ok=True)
freeze(OUT/'pilot.json',PILOT)
ranking.to_csv(OUT/'selection.csv',index=False)
display(ranking);print('Pilot output:',OUT)


In [ ]:
# Human prefix references and adaptive control helpers.
import math

def prefix_curve(word_lists, minimum_support=15):
    """Empirical quantiles at absolute alphabetic-word position; no extrapolation."""
    import numpy as np
    trajectories=[]
    for words in word_lists:
        seen=set(); trajectory=[]
        for w,word in enumerate(words,1):
            seen.add(word.lower()); trajectory.append(len(seen)/w)
        trajectories.append(trajectory)
    curve={}
    for w in range(1,max(map(len,trajectories),default=0)+1):
        ratios=[t[w-1] for t in trajectories if len(t)>=w]
        n=len(ratios)
        entry=dict(W=w,reference_n=n,supported=n>=minimum_support)
        if entry['supported']:
            entry.update(zip(['H_h30','H_h10','H_median','H_a10'],
                map(float,np.quantile(ratios,[.2,.4,.5,.6],method='linear'))))
        curve[str(w)]=entry
    return curve

def reference_at(curve,w):
    return curve.get(str(w),dict(W=w,reference_n=0,supported=False))

def adaptive_mode(previous,V,W,warmup_words,ref):
    if W<warmup_words or W==0: return 'normal'
    if not ref['supported']: return 'unsupported'
    k=V/W
    if previous=='increase' and k<ref['H_h10']: return 'increase'
    if previous=='decrease' and k>ref['H_h10']: return 'decrease'
    if k<ref['H_h30']: return 'increase'
    if k>ref['H_a10']: return 'decrease'
    return 'normal'

def adaptive_theta(mode,k,w,ref,schedules):
    if mode not in ('increase','decrease'): return None,None
    width=(ref['H_a10']-ref['H_h10'] if mode=='decrease'
           else ref['H_h10']-ref['H_h30'])
    q=max(0.,(k-ref['H_h10'] if mode=='decrease' else ref['H_h10']-k)/max(width,1/w))
    s=schedules[mode]
    assert 0<s['minimum']<=s['maximum']<=1 and s['slope']>0
    z=s['slope']*(q-s['midpoint'])
    sigmoid=math.exp(-z)/(1+math.exp(-z)) if z>=0 else 1/(1+math.exp(z))
    return float(s['minimum']+(s['maximum']-s['minimum'])*sigmoid),float(q)

def trajectory_metrics(trajectory,warmup_words):
    # Include word warmup+1 onward: these are the words eligible for intervention.
    eligible=[t for t in trajectory if t['W_after']>warmup_words]
    supported=[t for t in eligible if t['reference_after']['supported']]
    distances=[]; normalized=[]; longest=run=0
    for t in eligible:
        r=t['reference_after']
        if not r['supported']:
            run=0;continue
        k=t['k_after']; d=max(r['H_h30']-k,0.,k-r['H_a10'])
        distances.append(d)
        normalized.append(d/max(r['H_a10']-r['H_h30'],1/t['W_after']))
        run=run+1 if d>0 else 0;longest=max(longest,run)
    n=len(supported)
    return dict(post_warmup_words=len(eligible),supported_words=n,
        unsupported_words=len(eligible)-n,
        reference_coverage=n/len(eligible) if eligible else None,
        fraction_in_band=sum(d==0 for d in distances)/n if n else None,
        mean_band_distance=sum(distances)/n if n else None,
        mean_normalized_band_distance=sum(normalized)/n if n else None,
        maximum_band_distance=max(distances) if n else None,
        longest_outside_run=longest if n else None)

import torch
import spacy
from transformers import AutoTokenizer, AutoModelForCausalLM
nlp = spacy.load('en_core_web_sm')

def word_spans(text):
    # C02e2 normalizes newlines to spaces before feature extraction; preserve offsets.
    normalized = text.translate(str.maketrans({'\n':' ', '\r':' '}))
    return [(t.text,t.idx,t.idx+len(t.text)) for t in nlp.make_doc(normalized) if t.is_alpha]

def counts(text):
    words = [w.lower() for w,_,_ in word_spans(text)]
    return len(set(words)),len(words)



In [ ]:
# Word feedback engine. Pure Python except for the supplied generator/tokenizer backend.
import math
import heapq
from collections import Counter

class IncompleteGeneration(RuntimeError):
    """Operational or boundary failure; never interpreted as exhausted alternatives."""


def first_closed_word(backend, ids, fixed_prefix):
    """Return first fully delimited alphabetic spaCy token after fixed_prefix.
    Wait for whitespace (or EOS) to stabilize tokenization of contractions and
    punctuation inside the current chunk. A model token may span several words.
    Only the first word is committed; later characters remain tentative.
    """
    text = backend.decode(tuple(ids))
    stable = text.rstrip('\ufffd')
    common = min(len(stable), len(fixed_prefix))
    if stable[:common] != fixed_prefix[:common]:
        return dict(kind='prefix_conflict', text=text)
    if len(stable) < len(fixed_prefix):
        if backend.ended(ids):
            return dict(kind='prefix_conflict', text=text)
        return dict(kind='pending', text=text)
    final = backend.ended(ids)
    if final:
        if '\ufffd' in text:
            raise IncompleteGeneration('EOS with undecodable byte sequence.')
        closed_end = len(text)
    else:
        closed_end = max((i+1 for i,c in enumerate(stable) if c.isspace()), default=0)
    if closed_end < len(fixed_prefix):
        return dict(kind='pending', text=text)
    for word, start, end in backend.word_spans(text[:closed_end]):
        if start < len(fixed_prefix) < end:
            raise IncompleteGeneration('Tokenizer changed a previously committed word boundary.')
        if start >= len(fixed_prefix):
            return dict(kind='word', word=word, start=start, end=end, text=text)
    return dict(kind='end' if final else 'pending', text=text)


def greedy_to_word(backend, ids, probabilities, fixed_prefix):
    ids, probabilities = list(ids), list(probabilities)
    while True:
        view = first_closed_word(backend, ids, fixed_prefix)
        if view['kind'] in {'word', 'end'}:
            return dict(ids=ids, probabilities=probabilities, view=view)
        if view['kind'] == 'prefix_conflict':
            raise IncompleteGeneration('Ordinary decoding changed committed text.')
        choices = backend.choices(tuple(ids), threshold=None)
        if not choices:
            raise IncompleteGeneration('No permitted next token in ordinary decoding.')
        token, probability = choices[0]
        ids.append(int(token)); probabilities.append(float(probability))


def backtrack_anchor(backend, baseline_ids, fixed_prefix):
    # Find a token boundary at or before the character boundary. If a token
    # straddles an earlier committed word and the new word, replay that token
    # while constraining its already-committed characters exactly.
    for n in range(len(baseline_ids), -1, -1):
        prefix_ids = tuple(baseline_ids[:n])
        if backend.ended(prefix_ids):
            continue
        text = backend.decode(prefix_ids)
        if '\ufffd' not in text and fixed_prefix.startswith(text):
            return n
    raise IncompleteGeneration('No valid token checkpoint for this word.')


def search_word(backend, baseline, fixed_prefix, seen, desired, theta,
                on_search=lambda event: None):
    """Best-first search over paths whose word-forming tokens have raw
    probability >= theta. Greedy boundary-only confirmation is exempt. At most MAX_COMPLETED_CANDIDATES completed candidate paths per search.
    Completed undesired words terminate their branch. An operational failure
    propagates rather than masquerading as 'no alternative'.
    """
    if desired not in {'increase', 'decrease'} or not 0 < theta <= 1:
        raise ValueError('Invalid search mode or probability threshold.')
    anchor = backtrack_anchor(backend, baseline['ids'], fixed_prefix)
    root = tuple(baseline['ids'][:anchor])
    heap = [(0.0, 0, (), ())]
    serial = 0; expanded = 0; completed = 0; distinct=set()
    while heap:
        negative_logp, _, suffix, probs = heapq.heappop(heap)
        ids = root + suffix
        view = first_closed_word(backend, ids, fixed_prefix)
        if view['kind'] == 'prefix_conflict':
            continue
        if view['kind'] == 'word':
            novel = view['word'].lower() not in seen
            acceptable = novel if desired == 'increase' else not novel
            completed += 1; distinct.add(view['word'].lower())
            on_search(dict(event='completed_candidate', word=view['word'], novel=novel,
                acceptable=acceptable, branch_log_probability=-negative_logp,
                branch_tokens=list(suffix), branch_probabilities=list(probs),
                expanded=expanded, frontier=len(heap)))
            if acceptable:
                return dict(ids=list(ids), probabilities=baseline['probabilities'][:anchor]+list(probs),
                    view=view, anchor=anchor, branch_tokens=list(suffix),
                    branch_probabilities=list(probs), expanded=expanded,
                    completed_candidates=completed, distinct_candidates=len(distinct), exhausted=False,
                    boundary_exempt_indices=[i for i,p in enumerate(probs) if p<theta])
            if completed >= MAX_COMPLETED_CANDIDATES:
                on_search(dict(event='candidate_limit', completed_candidates=completed, expanded=expanded, frontier=len(heap)))
                return dict(exhausted=True, search_exhausted=False, fallback_reason='candidate_limit',
                    expanded=expanded, completed_candidates=completed, distinct_candidates=len(distinct), anchor=anchor)
            continue
        if view['kind'] == 'end':
            continue
        # Full distribution is filtered ONLY by the declared probability threshold
        # and the fixed exclusion of non-EOS protocol/control tokens.
        choices = backend.choices(ids, threshold=theta)
        # Boundary-only confirmation may follow the ordinary greedy token below
        # theta. It must close an already-spelled word without changing its letters.
        greedy = backend.choices(ids, threshold=None)
        exempt_token=None
        if greedy and greedy[0][1]<theta:
            token,p=greedy[0]
            before_text=backend.decode(ids)
            after_view=first_closed_word(backend, ids+(int(token),), fixed_prefix)
            boundary_only=(after_view['kind']=='word'
                and after_view['end']<=len(before_text)
                and before_text[:after_view['end']]==after_view['text'][:after_view['end']]
                and '\ufffd' not in before_text)
            if boundary_only:
                choices=choices+[(token,p)]; exempt_token=token
        on_search(dict(event='eligibility',theta=theta,
            eligible_lexical_or_boundary_tokens=len(choices)-int(exempt_token is not None),
            greedy_probability=greedy[0][1] if greedy else None,
            greedy_below_threshold=bool(greedy and greedy[0][1]<theta),
            boundary_exemption=exempt_token is not None))
        expanded += 1
        for token, probability in choices:
            assert probability >= theta or token==exempt_token
            serial += 1
            heapq.heappush(heap, (negative_logp-math.log(probability), serial,
                suffix+(int(token),), probs+(float(probability),)))
        on_search(dict(event='expanded', expanded=expanded, frontier=len(heap),
                       eligible_children=len(choices)))
    return dict(exhausted=True, expanded=expanded, completed_candidates=completed, distinct_candidates=len(distinct), anchor=anchor)


def initial_state(condition):
    return dict(condition=condition, ids=[], probabilities=[], committed_text='',
                words=[], mode='normal', trajectory=[], finished=False, status='ready',
                search_attempts=0, replacements=0, fallbacks=0)


def run_feedback(backend, state, curve, warmup_words, schedules,
                 checkpoint=lambda state: None, on_search=lambda event: None,
                 on_status=lambda state: None):
    # state is only mutated after a complete word is selected. During search it
    # stays at the last valid checkpoint; interruption cannot commit half a word.
    try:
        while not state['finished']:
            W = len(state['words']); seen = set(state['words']); V = len(seen)
            ref=reference_at(curve,W)
            mode=adaptive_mode(state['mode'],V,W,warmup_words,ref)
            if state['condition'] == 'baseline': mode = 'normal'
            theta,Q=adaptive_theta(mode,V/W if W else 0.,W,ref,schedules)
            ordinary = greedy_to_word(backend, state['ids'], state['probabilities'], state['committed_text'])
            if ordinary['view']['kind'] == 'end':
                state.update(ids=ordinary['ids'], probabilities=ordinary['probabilities'],
                    committed_text=ordinary['view']['text'], finished=True, status='completed_eos')
                actual = [w.lower() for w,_,_ in backend.word_spans(state['committed_text'])]
                if actual != state['words']:
                    state['finished'] = False
                    raise IncompleteGeneration('Final tokenizer counts differ from the committed-word ledger.')
                state.pop('last_error', None)
                checkpoint(state); on_status(state); break
            original_word = ordinary['view']['word']
            original_novel = original_word.lower() not in seen
            conflict = (mode == 'increase' and not original_novel) or (mode == 'decrease' and original_novel)
            selected = ordinary; action = 'ordinary'; info = {}
            if conflict:
                # Freeze all preceding text, including intervening punctuation.
                fixed_prefix = ordinary['view']['text'][:ordinary['view']['start']]
                info = search_word(backend, ordinary, fixed_prefix, seen, mode, theta, on_search)
                if info['exhausted']:
                    action = 'fallback_exhausted'
                else:
                    selected = info; action = 'backtracked'
            word = selected['view']['word']; novel = word.lower() not in seen
            if action == 'backtracked':
                assert novel == (mode == 'increase')
                assert all(p>=theta or i in info['boundary_exempt_indices']
                    for i,p in enumerate(info['branch_probabilities']))
            # No earlier committed text can change, including model-token straddles.
            new_prefix = selected['view']['text'][:selected['view']['end']]
            assert new_prefix.startswith(state['committed_text'])
            if conflict: assert new_prefix.startswith(fixed_prefix)
            entry = dict(fallback_reason=info.get('fallback_reason', 'exhausted' if action=='fallback_exhausted' else None), word_index=W+1, word=word, original_word=original_word,
                original_novel=original_novel, chosen_novel=novel, action=action,
                mode_before=mode, theta=theta, Q=Q, reference_before=ref,
                reference_after=reference_at(curve,W+1),
                boundary_exempt_indices=info.get('boundary_exempt_indices',[]),
                distinct_candidates=info.get('distinct_candidates',0), W_before=W, V_before=V, k_before=V/W if W else None,
                W_after=W+1, V_after=V+int(novel), k_after=(V+int(novel))/(W+1),
                search_expanded=info.get('expanded',0),
                completed_candidates=info.get('completed_candidates',0),
                branch_tokens=info.get('branch_tokens',[]),
                branch_probabilities=info.get('branch_probabilities',[]))
            state.update(ids=selected['ids'], probabilities=selected['probabilities'],
                         committed_text=new_prefix, mode=mode, status='running')
            state['words'].append(word.lower()); state['trajectory'].append(entry)
            state['search_attempts'] += int(conflict)
            state['replacements'] += int(action == 'backtracked')
            state['fallbacks'] += int(action == 'fallback_exhausted')
            checkpoint(state); on_status(state)
    except BaseException as exc:
        state['finished'] = False
        state['status'] = 'incomplete_interrupted' if isinstance(exc,KeyboardInterrupt) else 'incomplete_operational'
        state['last_error'] = type(exc).__name__ + ': ' + str(exc)
        checkpoint(state)
        raise
    return state

In [ ]:
torch.manual_seed(0)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = (torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
         else torch.float16 if DEVICE=='cuda' else torch.float32)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=(parent_contract.get('tokenizer_revision') or MODEL_REVISION))
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,dtype=DTYPE).to(DEVICE).eval()
EOS_IDS=model.generation_config.eos_token_id
EOS_IDS=set(EOS_IDS if isinstance(EOS_IDS,(list,tuple)) else [EOS_IDS])
EOS_IDS.discard(None)
if tokenizer.eos_token_id is not None: EOS_IDS.add(tokenizer.eos_token_id)
assert EOS_IDS
FORBIDDEN_SPECIAL=set(tokenizer.all_special_ids)-EOS_IDS
CONTEXT_SIZE=int(model.config.max_position_embeddings)
KEEP_LAST='logits_to_keep' in inspect.signature(model.forward).parameters

PROMPT_VERSION='meaning-preserving-rewrite-v1'
def messages_for(case):
    return [dict(role='system',content='Rewrite the supplied passage. Preserve its meaning, facts, names, numbers, and genre. Use approximately the same length. Return only the rewritten passage, with no explanation or preamble.'),
            dict(role='user',content=case['original'])]

class LocalBackend:
    def __init__(self,prompt_ids): self.prompt_ids=list(prompt_ids)
    def decode(self,ids):
        return tokenizer.decode(list(ids),skip_special_tokens=True,clean_up_tokenization_spaces=False)
    def ended(self,ids): return bool(ids) and ids[-1] in EOS_IDS
    def word_spans(self,text): return word_spans(text)
    def choices(self,ids,threshold=None):
        full=self.prompt_ids+list(ids)
        if len(full)>=CONTEXT_SIZE:
            raise IncompleteGeneration(f'Model context capacity reached ({CONTEXT_SIZE}); search or generation is incomplete.')
        x=torch.tensor([full],dtype=torch.long,device=DEVICE)
        with torch.inference_mode():
            kwargs={'logits_to_keep':1} if KEEP_LAST else {}
            output=model(input_ids=x,attention_mask=torch.ones_like(x),use_cache=False,**kwargs)
            probabilities=torch.softmax(output.logits[0,-1].float(),dim=-1)
            if not torch.isfinite(probabilities).all().item():
                raise IncompleteGeneration('Non-finite next-token distribution.')
            valid=probabilities.clone()
            for i in FORBIDDEN_SPECIAL:
                if i<len(valid): valid[i]=-1
            if threshold is None:
                ids_out=[int(valid.argmax().item())]
            else:
                ids_out=torch.nonzero(valid>=threshold,as_tuple=False).flatten().tolist()
            pairs=[(int(i),float(probabilities[i].item())) for i in ids_out]
        return sorted(pairs,key=lambda pair:(-pair[1],pair[0]))

from collections.abc import Mapping
from numbers import Integral

def normalized_prompt_ids(encoded):
    if isinstance(encoded,Mapping): encoded=encoded['input_ids']
    if hasattr(encoded,'tolist'): encoded=encoded.tolist()
    if isinstance(encoded,(list,tuple)) and len(encoded)==1:
        first=encoded[0]
        if hasattr(first,'tolist'): first=first.tolist()
        if isinstance(first,(list,tuple)): encoded=first
    if not isinstance(encoded,(list,tuple)) or not encoded:
        raise TypeError('Expected nonempty token IDs.')
    if not all(isinstance(t,Integral) and not isinstance(t,bool) for t in encoded):
        raise TypeError('Expected one flat integer token sequence.')
    ids=[int(t) for t in encoded]
    assert all(0<=t<model.config.vocab_size for t in ids)
    return ids



In [ ]:
# Run after the engine-definition cell, before the generation cell.
SEARCH_BUDGET = {
    "version": "word_search_budget_pilot_v1",
    "max_expanded": 1000,
    "max_frontier": 10000,
}

budget_path = OUT / "word_search_budget.json"
if budget_path.exists():
    assert json.loads(budget_path.read_text()) == SEARCH_BUDGET, (
        "Search-budget settings changed; retain separate provenance."
    )
else:
    write_json(budget_path, SEARCH_BUDGET)

# Preserve the original function when this patch cell is rerun.
_search_word_before_budget = search_word


class WordSearchBudgetReached(Exception):
    pass


def search_word(
    backend, baseline, fixed_prefix, seen, desired, theta,
    on_search=lambda event: None,
):
    progress = {
        "expanded": 0,
        "frontier": 0,
        "completed_candidates": 0,
    }
    distinct = set()

    def monitored_search(event):
        if event["event"] == "completed_candidate":
            progress["completed_candidates"] += 1
            distinct.add(event["word"].lower())

        for key in ("expanded", "frontier"):
            if key in event:
                progress[key] = int(event[key])

        on_search(event)

        # Check after expansion, when the updated queue size is available.
        if event["event"] == "expanded":
            if (
                progress["expanded"] >= SEARCH_BUDGET["max_expanded"]
                or progress["frontier"] >= SEARCH_BUDGET["max_frontier"]
            ):
                raise WordSearchBudgetReached()

    try:
        return _search_word_before_budget(
            backend, baseline, fixed_prefix, seen, desired, theta,
            monitored_search,
        )
    except WordSearchBudgetReached:
        on_search({
            "event": "search_budget",
            "budget_version": SEARCH_BUDGET["version"],
            **progress,
            "distinct_candidates": len(distinct),
            "search_exhausted": False,
        })

        # The legacy caller uses `exhausted` to select ordinary fallback.
        # `search_exhausted=False` records that alternatives still existed.
        return {
            "exhausted": True,
            "search_exhausted": False,
            "fallback_reason": "search_budget",
            "expanded": progress["expanded"],
            "completed_candidates": progress["completed_candidates"],
            "distinct_candidates": len(distinct),
        }

budgeted_search = search_word
# Replace the generation/run cell with this cell.
# Keep the existing engine and LocalBackend definitions.

RUNAWAY_POLICY = {
    "version": "runaway_guard_v1",
    "max_source_word_ratio": 2.0,
    "minimum_repeated_block_words": 12,
    "repeated_copies": 3,
}
GUARD_HASH = digest(RUNAWAY_POLICY)
policy_path = OUT / "runaway_policy.json"

if policy_path.exists():
    assert digest(json.loads(policy_path.read_text())) == GUARD_HASH, (
        "Guard settings changed. Preserve the previous experiment before changing them."
    )
else:
    write_json(policy_path, RUNAWAY_POLICY)


class RunawayGeneration(IncompleteGeneration):
    pass


def repeated_suffix(words):
    """Detect exact, consecutive repeated blocks in the committed word ledger."""
    copies = RUNAWAY_POLICY["repeated_copies"]
    minimum = RUNAWAY_POLICY["minimum_repeated_block_words"]
    for width in range(minimum, len(words) // copies + 1):
        block = words[-width:]
        if all(
            words[len(words) - (j + 1) * width:
                  len(words) - j * width] == block
            for j in range(1, copies)
        ):
            return {
                "block_words": width,
                "copies": copies,
                "block": list(block),
            }
    return None


def runaway_reason(s, c):
    # Check repetition first so an already-long checkpoint gets a useful diagnosis.
    repeated = repeated_suffix(s["words"])
    if repeated is not None:
        return {"reason": "repeated_block", **repeated}

    limit = math.ceil(
        c["W_ref"] * RUNAWAY_POLICY["max_source_word_ratio"]
    )
    if len(s["words"]) >= limit and not s["finished"]:
        return {
            "reason": "source_length_limit",
            "generated_words": len(s["words"]),
            "limit_words": limit,
        }
    return None


def show_repetition_onset(s):
    words = s["words"]
    for end in range(
        RUNAWAY_POLICY["minimum_repeated_block_words"]
        * RUNAWAY_POLICY["repeated_copies"],
        len(words) + 1,
    ):
        hit = repeated_suffix(words[:end])
        if hit is None:
            continue

        start = end - hit["copies"] * hit["block_words"] + 1
        print(
            f"First detected triple repetition: words {start}–{end}; "
            f"block length {hit['block_words']}."
        )
        print("Repeated block:", " ".join(hit["block"]))

        # This identifies nearby interventions; it does not establish causation.
        trajectory = pd.DataFrame(s["trajectory"])
        nearby = trajectory.loc[
            trajectory.word_index.between(max(1, start - 10), end)
        ]
        display(nearby[
            ["word_index", "word", "mode_before", "action", "k_after"]
        ])
        return

    print("No exact triple block detected in the committed word ledger.")




In [ ]:
# Preserve upward behavior and existing hysteresis; start decrease at the median.
_original_mode=adaptive_mode
_original_theta=adaptive_theta
def adaptive_mode(previous,V,W,warmup_words,ref):
    mode=_original_mode(previous,V,W,warmup_words,ref)
    if W>=warmup_words and W>0 and ref['supported'] and V/W>ref['H_median']:
        return 'decrease'
    return mode

def adaptive_theta(mode,k,w,ref,schedules):
    theta,q=_original_theta(mode,k,w,ref,schedules)
    if mode=='decrease':theta*=PILOT['downward_theta_multiplier']
    return theta,q

for c in cases:
    prompt_ids=normalized_prompt_ids(tokenizer.apply_chat_template(messages_for(c),tokenize=True,add_generation_prompt=True,return_dict=True,return_tensors=None))
    assert prompt_ids==c['prompt_ids'],f"Prompt mismatch: {c['case_id']}"
    assert counts(c['original'])[1]==c['W_ref']
assert nlp.meta==parent_contract['spacy_meta'],'spaCy pipeline changed'
IMPLEMENTATION_SHA256=NOTEBOOK_CODE_HASH


In [ ]:
def word_path_logp(backend, result, anchor):
    target = result['view']['text'][:result['view']['end']]
    for n in range(anchor+1, len(result['ids'])+1):
        decoded = backend.decode(result['ids'][:n])
        if '\ufffd' not in decoded and decoded.startswith(target):
            probs = result['probabilities'][anchor:n]
            assert probs and all(0 < x <= 1 for x in probs)
            return sum(math.log(x) for x in probs)
    raise IncompleteGeneration('No completed-word scoring boundary.')

def make_search(ratio_floor):
    def guarded(backend, baseline, fixed_prefix, seen, desired, theta,
                on_search=lambda e: None):
        candidate = budgeted_search(
            backend, baseline, fixed_prefix, seen, desired, theta, on_search)
        if candidate['exhausted'] or ratio_floor is None:
            return candidate
        anchor = candidate['anchor']
        ordinary_logp = word_path_logp(backend, baseline, anchor)
        candidate_logp = word_path_logp(backend, candidate, anchor)
        log_ratio = candidate_logp - ordinary_logp
        accepted = log_ratio >= math.log(ratio_floor)
        on_search(dict(event='relative_likelihood_guard',
            ordinary_word=baseline['view']['word'],
            candidate_word=candidate['view']['word'],
            ordinary_logp=ordinary_logp, candidate_logp=candidate_logp,
            log_ratio=log_ratio, ratio_floor=ratio_floor, accepted=accepted))
        if accepted:
            return candidate
        return dict(exhausted=True, search_exhausted=False,
            fallback_reason='relative_likelihood_rejected',
            expanded=candidate.get('expanded',0),
            completed_candidates=candidate.get('completed_candidates',0),
            distinct_candidates=candidate.get('distinct_candidates',0))
    return guarded

guard_implementation = hashlib.sha256(
    (inspect.getsource(word_path_logp)+inspect.getsource(make_search)).encode()
).hexdigest()
RUN_CONTRACT = dict(
    version='C02w2_top5_v1', pilot=PILOT, notebook_code_hash=NOTEBOOK_CODE_HASH, parent_hash=digest(parent_contract),
    ids=sorted(selected_ids), arms=ARMS,
    implementation=IMPLEMENTATION_SHA256, guard_implementation=guard_implementation,
    model_revision=getattr(model.config,'_commit_hash',None),
    tokenizer_revision=tokenizer.init_kwargs.get('_commit_hash'),
    versions={p:metadata.version(p) for p in ['torch','transformers','spacy','numpy','pandas']},
    dtype=str(DTYPE), device=DEVICE,
    search_budget=SEARCH_BUDGET, runaway_policy=RUNAWAY_POLICY,
    rejection_policy='first_candidate_rejected_then_ordinary',
)
freeze(OUT/'run_contract.json', RUN_CONTRACT)
RUN_HASH = digest(RUN_CONTRACT)


## Regenerate the five selected passages
Progress prints during search; every committed word is checkpointed. An interrupted run resumes from its checkpoint. A completed or failed run is reported without restarting it automatically. Larger searches can take longer on the same hardware.


In [ ]:
generation_rows = []
for case_index, c in enumerate(cases):
    arm_names = list(ARMS)
    arm_names = arm_names[case_index % 4:] + arm_names[:case_index % 4]
    for arm in arm_names:
        folder = OUT/c['case_id']/arm
        folder.mkdir(parents=True, exist_ok=True)
        path = folder/'state.json'
        seed_condition = 'baseline' if arm == 'ordinary' else 'controlled'
        elapsed_before = 0.
        if path.exists():
            payload = json.loads(path.read_text())
            assert payload['run_hash'] == RUN_HASH
            state = payload['state']
            elapsed_before = payload.get('active_seconds',0.)
        else:
            state = initial_state(seed_condition)
        terminal = state['status']=='completed_eos' or state['status'].startswith('failed_')
        start = time.monotonic()
        last_report = [0.]
        attempt_id = uuid.uuid4().hex
        def save(s):
            write_json(path, dict(run_hash=RUN_HASH, state=s,
                active_seconds=elapsed_before+time.monotonic()-start))
        def checkpoint(s):
            reason = runaway_reason(s,c)
            if reason:
                s['stop_reason'] = reason
                save(s)
                raise RunawayGeneration(str(reason))
            save(s)
        def report(s):
            now = time.monotonic()
            if now-last_report[0]>=10:
                print(c['case_id'],arm,'words',len(s['words']),
                      'mode',s['mode'],'edits',s['replacements'],
                      'fallbacks',s['fallbacks'],flush=True)
                last_report[0]=now
        if not terminal:
            print('RUN',c['case_id'],arm,flush=True)
            backend = LocalBackend(c['prompt_ids'])
            search_word = make_search(ARMS[arm])
            try:
                with (folder/'search_audit.jsonl').open('a',buffering=1) as audit:
                    def search_log(e):
                        audit.write(json.dumps(dict(attempt_id=attempt_id,
                            word_index=len(state['words'])+1,**e))+'\n')
                        report(state)
                    run_feedback(backend,state,c['curve'],c['warmup_words'],
                                 THETA_SCHEDULES,checkpoint,search_log,report)
            except RunawayGeneration:
                state.update(finished=False,status='failed_runaway')
                save(state)
            except KeyboardInterrupt:
                save(state)
                print('Interrupted; rerun this cell to resume:',folder)
                raise
            except torch.cuda.OutOfMemoryError:
                save(state)
                raise
            except IncompleteGeneration as exc:
                state.update(finished=False,status='failed_operational',
                             last_error=str(exc))
                save(state)
            finally:
                search_word = budgeted_search
        (folder/'text.txt').write_text(state['committed_text'])
        pd.DataFrame(state['trajectory']).to_csv(folder/'word_trajectory.csv',index=False)
        generation_rows.append(dict(case_id=c['case_id'],arm=arm,status=state['status'],
            words=len(state['words']),reused=terminal))
        pd.DataFrame(generation_rows).to_csv(OUT/'generation_outcomes.csv',index=False)
display(pd.DataFrame(generation_rows).groupby(['arm','status']).size())


In [ ]:
rows=[];reviews=[]
for c in cases:
    cid=c['case_id']
    for label,path in [('Previous 0.05',PREV/cid/'ratio_005/state.json'),('Stronger control',OUT/cid/'robust_005/state.json')]:
        s=json.loads(path.read_text())['state'];tr=s['trajectory']
        ref=reference_at(c['curve'],len(s['words']))
        k=len(set(s['words']))/len(s['words']) if s['words'] else np.nan
        supported=ref['supported']
        reasons=Counter(t.get('fallback_reason') for t in tr if t.get('fallback_reason'))
        metrics=trajectory_metrics(tr,c['warmup_words'])
        rows.append(dict(case_id=cid,condition=label,status=s['status'],unique=k,words=len(s['words']),
            target_low=ref.get('H_h30',np.nan),target_high=ref.get('H_a10',np.nan),
            final_target_status=('unsupported' if not supported else 'above' if k>ref['H_a10'] else 'below' if k<ref['H_h30'] else 'in_band'),
            excess=max(0.,k-ref['H_a10']) if supported else np.nan,
            replacements=s['replacements'],search_attempts=s['search_attempts'],fallbacks=s['fallbacks'],
            fallback_reasons=json.dumps(dict(reasons)),**metrics))
        if label=='Stronger control':
            reviews.append(dict(case_id=cid,source=c['original'],text=s['committed_text'],status=s['status'],fidelity_review='',grammar_review='',repetition_review=''))
summary=pd.DataFrame(rows);summary.to_csv(OUT/'target_comparison.csv',index=False)
pd.DataFrame(reviews).to_csv(OUT/'pilot_texts.csv',index=False)
if not (OUT/'manual_review.csv').exists():pd.DataFrame(reviews).to_csv(OUT/'manual_review.csv',index=False)
display(summary)
display(pd.crosstab(summary.condition,summary.final_target_status))
fig,ax=plt.subplots(figsize=(8,4))
for label in summary.condition.unique():
    z=summary.loc[summary.condition.eq(label)].set_index('case_id').loc[ranking.case_id]
    ax.plot(np.arange(5),z.excess,'o-',label=label)
ax.set_xticks(np.arange(5),ranking.case_id,rotation=30)
ax.set_ylabel('Final excess above own supported target');ax.legend();fig.tight_layout()
for ext in ['png','pdf']:_export_figure(fig,OUT/f'target_excess.{ext}',dpi=300,bbox_inches='tight')
plt.show()
print('Pilot complete. Inspect target_comparison.csv and manual_review.csv before downstream processing. Existing 54 outputs unchanged.')
